# STAC Catalog Builder — Salinity Increase

This notebook generates a STAC catalog for preprocessed salinity increase COG datasets. It follows the same workflow as `13_salinity_template_C.ipynb`, with paths and metadata configured for the `salinity_increase` product.

## Pipeline overview

| Step | Purpose |
|------|---------|
| 1. Configure paths and options | Define input/output locations, scenario folders, and cloud targets |
| 2. Define STAC helper functions | Provide reusable builders for collections, items, and assets |
| 3. Create the STAC collection | Instantiate the dataset-level catalog container from the metadata JSON |
| 4. Build one test STAC item | Validate item construction on a single COG before batch processing |
| 5. Build all STAC items | Generate and register one STAC item per COG across all scenarios |
| 6. Save STAC catalog locally | Persist the collection and items to `STAC/data/current/` |
| 7. Upload COGs to Google Cloud | Publish raster assets to GCS (optional) |
| 8. Upload STAC catalog to Google Cloud | Publish the catalog JSON to GCS (optional) |
| 9. Publish layers to GeoServer WMS | Register COGs as WMS layers on GeoServer (optional) |

## Prerequisites

Run `11_Salinity_preprocessing.ipynb` to produce the salinity increase COG inputs and `metadata_salinity_increase.json` before executing this template.

In [1]:
import datetime
import json
import os
import time
import warnings
from pathlib import Path
from posixpath import join as urljoin

import pandas as pd
import pystac
import pystac_client
import rasterio
import requests
import rioxarray
import shapely
import urllib3
import xarray as xr
from dotenv import load_dotenv
from geo.Geoserver import Geoserver
from pystac.extensions import eo, raster
from pystac.stac_io import DefaultStacIO
from stactools.core.utils import antimeridian

from coclicodata.coclico_stac.layouts import CoCliCoCOGLayout
from coclicodata.etl.cloud_utils import dir_to_google_cloud, load_google_credentials

## 1) Configure paths and options

Define local and cloud paths, then load the metadata JSON for salinity increase. Processing parameters (`SPATIAL_RESOLUTION`, `CRS`, `COLLECTION_ID`, etc.) must be present in the metadata file — missing keys raise an error.

Metadata file: `N:\...\stac_folder\salinity_increase\metadata_salinity_increase.json`

For step 9 (GeoServer WMS), copy `global-coastal-atlas/.env.example` to `global-coastal-atlas/.env` and set `GEOSERVER_USERNAME` and `GEOSERVER_PASSWORD`.

Scenario folders are discovered automatically from `cog_dir`. Salinity increase has no `baseline` folder — only climate scenarios. A fallback list is used when no subdirectories are found.

In [2]:
repo_root = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")
stac_dir = repo_root / "global-coastal-atlas/STAC/data/current"

data_dir = Path(
    r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity_increase"
)
cog_dir = data_dir / "cogs"
metadata_path = data_dir / "metadata_salinity_increase.json"
google_cred_path = Path(
    r"N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\google_credentials.json"
)

fallback_scenarios = [
    "cc45y",
    "cc85y",
    "cc85sb2y",
    "cc45sm2y",
    "cc45sm2rb1y",
    "cc85sb2rb3y",
]

gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"

env_path = repo_root / "global-coastal-atlas/.env"
geoserver_gcs_mount_prefix = "/opt/gca-data-public/gca"

if not cog_dir.exists():
    raise FileNotFoundError(f"COG directory not found: {cog_dir}")

with open(metadata_path) as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "SPATIAL_RESOLUTION",
    "CRS",
    "ITEM_BBOX_CRS",
    "NODATA",
    "DATA_TYPE",
    "COLLECTION_ID",
    "WMS_DATASET",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(
        f"Missing required metadata keys in {metadata_path}: {', '.join(missing_keys)}"
    )

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
spatial_resolution = metadata["SPATIAL_RESOLUTION"]
href_prefix = urljoin(gcs_protocol, bucket_name, bucket_proj, proj_name)

discovered_scenarios = sorted(p.name for p in cog_dir.iterdir() if p.is_dir())
scenarios = discovered_scenarios or fallback_scenarios

print("COG input:", cog_dir)
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Spatial resolution (m):", spatial_resolution)
print("Native CRS:", metadata["CRS"])
print("Item bbox CRS:", metadata["ITEM_BBOX_CRS"])
print("Scenarios:", scenarios)

COG input: N:\Deltabox\Postbox\Athanasiou, Panos\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\salinity_increase\cogs
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\salinity_increase
Collection id: salinity_increase
Spatial resolution (m): 2000
Native CRS: EPSG:32648
Item bbox CRS: EPSG:4326
Scenarios: ['cc45sm2rb1y', 'cc45sm2y', 'cc45y', 'cc85sb2y', 'cc85y']


## 2) Define STAC helper functions

Define the core building blocks for catalog construction. Functions take `metadata` (and `collection_id` where needed) as explicit arguments so they can be moved to a separate `.py` module later without relying on notebook globals.

- `create_collection(metadata, collection_id)` — assembles collection-level metadata and extensions
- `create_item(block, item_id, metadata)` — derives item geometry, bbox, and projection metadata from a raster
- `create_asset()` / `create_wms_asset()` — attach data and visual assets to an item
- `process_block(file_path, base_path, metadata)` — end-to-end item builder for a single COG
- `item_datetime_from_path()` — derives item datetime from the year suffix in the COG filename (e.g. `p50_2030.tif` → 2030-01-01)
- `get_paths()` — resolves scenario folder paths for iteration
- `configure_geoserver_ssl()` / `load_geoserver_credentials()` — GeoServer client setup
- `publish_layers_to_geoserver()` — publish COGs as external coverage stores and apply WMS styles

In [3]:
DATA_TYPE_MAP = {
    "float32": raster.DataType.FLOAT32,
    "float64": raster.DataType.FLOAT64,
    "int8": raster.DataType.INT8,
    "int16": raster.DataType.INT16,
    "int32": raster.DataType.INT32,
    "uint8": raster.DataType.UINT8,
    "uint16": raster.DataType.UINT16,
}


def get_raster_data_type(metadata: dict):
    data_type = metadata["DATA_TYPE"].lower()
    if data_type not in DATA_TYPE_MAP:
        raise ValueError(f"Unsupported DATA_TYPE in metadata: {metadata['DATA_TYPE']}")
    return DATA_TYPE_MAP[data_type]


def item_datetime_from_path(file_path: Path) -> pd.Timestamp:
    year_part = file_path.stem.split("_")[-1]
    year = int(year_part) if len(year_part) == 4 else 2000 + int(year_part)
    return pd.Timestamp(year, 1, 1)


def create_collection(metadata: dict, collection_id: str) -> pystac.Collection:
    license = metadata["LICENSE"]
    if "Creative Commons" in license and "4.0" in license:
        license = "CC-BY-4.0"

    providers = [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[
                pystac.provider.ProviderRole.PRODUCER,
                pystac.provider.ProviderRole.LICENSOR,
            ],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[
                pystac.provider.ProviderRole.PROCESSOR,
                pystac.provider.ProviderRole.HOST,
            ],
            url="https://deltares.nl",
        ),
    ]

    start_datetime = datetime.datetime.strptime(
        metadata["TEMPORAL_EXTENT"][0].split("T")[0], "%Y-%m-%d"
    )
    end_datetime = None
    if len(metadata["TEMPORAL_EXTENT"]) > 1 and metadata["TEMPORAL_EXTENT"][1]:
        end_datetime = datetime.datetime.strptime(
            metadata["TEMPORAL_EXTENT"][1].split("T")[0], "%Y-%m-%d"
        )

    extent = pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[start_datetime, end_datetime]]),
    )

    collection = pystac.Collection(
        id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        license=license,
        providers=providers,
        extent=extent,
        catalog_type=pystac.CatalogType.RELATIVE_PUBLISHED,
    )
    collection.keywords = metadata["KEYWORDS"]

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": pystac.MediaType.COG,
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["DESCRIPTION"],
            "xarray:storage_options": {"token": "google_default"},
        }
    }
    collection.extra_fields["deltares:units"] = metadata["UNITS"]

    return collection


def create_item(
    block,
    item_id: str,
    metadata: dict,
    antimeridian_strategy=antimeridian.Strategy.SPLIT,
):
    dst_crs = rasterio.crs.CRS.from_string(metadata["ITEM_BBOX_CRS"])

    bbox = rasterio.warp.transform_bounds(block.rio.crs, dst_crs, *block.rio.bounds())
    geometry = shapely.geometry.mapping(shapely.make_valid(shapely.geometry.box(*bbox)))
    bbox = shapely.make_valid(shapely.box(*bbox)).bounds

    item = pystac.Item(
        id=item_id,
        geometry=geometry,
        bbox=bbox,
        datetime=pd.Timestamp(block["time"].item()),
        properties={},
    )
    antimeridian.fix_item(item, antimeridian_strategy)
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)

    ext = pystac.extensions.projection.ProjectionExtension.ext(item, add_if_missing=True)
    ext.bbox = block.rio.bounds()
    ext.shape = tuple(v for k, v in block.sizes.items() if k in ["y", "x"])
    ext.epsg = block.rio.crs.to_epsg()
    ext.geometry = shapely.geometry.mapping(shapely.geometry.box(*ext.bbox))
    ext.transform = list(block.rio.transform())[:6]
    ext.add_to(item)

    item.properties["deltares:item_key"] = item_id
    return item


def create_asset(
    item,
    asset_title: str,
    asset_href: str,
    nodata,
    resolution,
    data_type,
    metadata: dict,
    nbytes=None,
):
    asset = pystac.Asset(
        href=asset_href,
        media_type=pystac.MediaType.COG,
        title=asset_title,
        roles=["data"],
    )
    item.add_asset("data", asset)
    pystac.extensions.file.FileExtension.ext(asset, add_if_missing=True)
    if nbytes:
        asset.extra_fields["file:size"] = nbytes
    raster.RasterExtension.ext(asset, add_if_missing=True).bands = [
        raster.RasterBand.create(
            nodata=nodata,
            spatial_resolution=resolution,
            data_type=data_type,
        )
    ]
    eo.EOExtension.ext(asset, add_if_missing=True).bands = [
        eo.Band.create(name=asset_title, description=metadata["DESCRIPTION"])
    ]
    return item


def create_wms_asset(item, asset_title: str, asset_href: str):
    asset = pystac.Asset(
        href=asset_href,
        media_type="application/png",
        title=asset_title,
        description="OGS WMS url",
        roles=["visual"],
    )
    item.add_asset("visual", asset)
    return item


def process_block(
    file_path: Path,
    base_path: Path,
    metadata: dict,
    storage_prefix: str = "",
) -> pystac.Item:
    block = xr.open_dataset(file_path, engine="rasterio", mask_and_scale=False)
    block = block.assign_coords(time=item_datetime_from_path(file_path).isoformat())

    item_id = file_path.relative_to(base_path).as_posix()
    item = create_item(block, item_id=item_id, metadata=metadata)
    resolution = metadata["SPATIAL_RESOLUTION"]
    data_type = get_raster_data_type(metadata)
    wms_dataset = metadata["WMS_DATASET"]

    for var in block:
        da = block[var]
        href = urljoin(storage_prefix, Path(file_path.name).as_posix())
        nbytes = os.path.getsize(file_path)
        nodata = da.rio.nodata.item() if da.rio.nodata is not None else metadata["NODATA"]

        item = create_asset(
            item,
            asset_title=f"{file_path.parent.name}_{file_path.stem}",
            asset_href=href,
            nodata=nodata,
            resolution=resolution,
            data_type=data_type,
            metadata=metadata,
            nbytes=nbytes,
        )
        item = create_wms_asset(
            item,
            asset_title=f"{file_path.parent.name}_{file_path.stem}",
            asset_href=f"https://international-delta-platform.avi.directory.intra/geoserver/wms/{wms_dataset}",
        )

    return item


def get_paths(folder_structure, base_dir=""):
    paths = []
    for key, value in folder_structure.items():
        if isinstance(value, dict):
            paths.extend(get_paths(value, os.path.join(base_dir, key)))
        elif isinstance(value, list):
            if value:
                for item in value:
                    if item != "":
                        paths.append(os.path.join(base_dir, key, item))
            else:
                paths.append(os.path.join(base_dir, key))
    return paths


def configure_geoserver_ssl() -> None:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    original_request = requests.Session.request

    def patched_request(self, method, url, **kwargs):
        kwargs.setdefault("verify", False)
        return original_request(self, method, url, **kwargs)

    requests.Session.request = patched_request


def load_geoserver_credentials(env_path: Path) -> dict[str, str]:
    if not env_path.exists():
        raise FileNotFoundError(
            f"GeoServer credentials file not found: {env_path}. "
            "Copy global-coastal-atlas/.env.example to global-coastal-atlas/.env and fill in the values."
        )

    load_dotenv(env_path)
    credentials = {
        "base_url": os.getenv(
            "GEOSERVER_URL",
            "https://international-delta-platform.avi.directory.intra/geoserver",
        ),
        "username": os.getenv("GEOSERVER_USERNAME"),
        "password": os.getenv("GEOSERVER_PASSWORD"),
    }
    missing = [key for key, value in credentials.items() if key != "base_url" and not value]
    if missing:
        raise KeyError(
            f"Missing GeoServer credentials in {env_path}: {', '.join(missing)}"
        )
    return credentials


def create_geoserver_client(base_url: str, username: str, password: str) -> Geoserver:
    configure_geoserver_ssl()
    return Geoserver(base_url, username=username, password=password)


def ensure_geoserver_workspace(geo: Geoserver, workspace: str) -> None:
    try:
        geo.create_workspace(workspace=workspace)
        print(f"Workspace '{workspace}' created")
    except Exception as exc:
        if "already exists" in str(exc).lower() or "409" in str(exc):
            print(f"Workspace '{workspace}' already exists")
        else:
            raise


def set_layer_default_style(
    base_url: str,
    workspace: str,
    layer_name: str,
    style_name: str,
    username: str,
    password: str,
) -> None:
    rest_url = f"{base_url}/rest/layers/{workspace}:{layer_name}.json"
    response = requests.put(
        rest_url,
        auth=(username, password),
        json={"layer": {"defaultStyle": {"name": style_name}}},
        verify=False,
        headers={"Content-Type": "application/json"},
    )
    if response.status_code not in (200, 201):
        raise RuntimeError(
            f"Could not set style for {layer_name}: {response.status_code} {response.text}"
        )


def wms_layer_name_from_cog(scenario: str, cog_path: Path) -> str:
    return f"{scenario}_{cog_path.stem}"


def wms_external_tif_path(
    metadata: dict,
    scenario: str,
    cog_path: Path,
    gcs_mount_prefix: str,
) -> str:
    dataset = metadata["WMS_DATASET"]
    return f"{gcs_mount_prefix}/{dataset}/{scenario}/{cog_path.name}"


def publish_cog_layer_to_geoserver(
    geo: Geoserver,
    metadata: dict,
    scenario: str,
    cog_path: Path,
    *,
    geoserver_base_url: str,
    geoserver_username: str,
    geoserver_password: str,
    gcs_mount_prefix: str,
    layer_style: str,
) -> dict:
    workspace = metadata["WMS_DATASET"]
    layer_name = wms_layer_name_from_cog(scenario, cog_path)
    tif_path = wms_external_tif_path(metadata, scenario, cog_path, gcs_mount_prefix)

    try:
        geo.delete_layer(workspace=workspace, layer_name=layer_name)
        geo.delete_coveragestore(workspace=workspace, store_name=layer_name)
    except Exception:
        pass

    geo.create_coveragestore(
        layer_name=layer_name,
        path=tif_path,
        workspace=workspace,
        method="external",
    )
    set_layer_default_style(
        base_url=geoserver_base_url,
        workspace=workspace,
        layer_name=layer_name,
        style_name=layer_style,
        username=geoserver_username,
        password=geoserver_password,
    )
    return {
        "scenario": scenario,
        "layer_name": layer_name,
        "workspace": workspace,
        "tif_path": tif_path,
        "status": "published",
    }


def publish_layers_to_geoserver(
    geo: Geoserver,
    metadata: dict,
    scenarios: list[str],
    cog_dir: Path,
    *,
    geoserver_base_url: str,
    geoserver_username: str,
    geoserver_password: str,
    gcs_mount_prefix: str,
    layer_style: str | None = None,
    sleep_seconds: float = 1.0,
) -> tuple[list[dict], list[dict]]:
    workspace = metadata["WMS_DATASET"]
    layer_style = layer_style or workspace
    ensure_geoserver_workspace(geo, workspace)

    published_rows = []
    error_rows = []

    for scenario in scenarios:
        scenario_dir = cog_dir / scenario
        if not scenario_dir.exists():
            error_rows.append({"scenario": scenario, "error": "folder not found"})
            continue

        print(f"Publishing WMS layers for scenario: {scenario}")
        for cog_path in sorted(scenario_dir.glob("*.tif")):
            try:
                row = publish_cog_layer_to_geoserver(
                    geo,
                    metadata,
                    scenario,
                    cog_path,
                    geoserver_base_url=geoserver_base_url,
                    geoserver_username=geoserver_username,
                    geoserver_password=geoserver_password,
                    gcs_mount_prefix=gcs_mount_prefix,
                    layer_style=layer_style,
                )
                published_rows.append(row)
                print(f"  Published {row['workspace']}:{row['layer_name']}")
                time.sleep(sleep_seconds)
            except Exception as exc:
                error_rows.append(
                    {
                        "scenario": scenario,
                        "file": cog_path.name,
                        "error": str(exc),
                    }
                )
                print(f"  Error for {scenario}/{cog_path.name}: {exc}")

    return published_rows, error_rows

## 3) Create the STAC collection

Instantiate the STAC collection that will contain all dataset items. A collection describes dataset-level metadata — title, description, license, providers, spatial and temporal extent, and keywords — rather than individual raster files.

This step is separated from item construction because collection metadata is sourced from the metadata JSON configured in step 1 and is independent of any single COG. The resulting `collection` object acts as the parent container into which items are added in step 5.

**Metadata fields used:** `PROVIDERS`, `DESCRIPTION`, `KEYWORDS`, `SPATIAL_EXTENT`, `TEMPORAL_EXTENT`, `LICENSE`, `TITLE`, `UNITS`, `COLLECTION_ID`

In [4]:
collection = create_collection(metadata, collection_id)

print(f"Collection ready: {collection.id}")
print(f"Description: {metadata['DESCRIPTION'][:80]}{'...' if len(metadata['DESCRIPTION']) > 80 else ''}")
print(f"Provider: {metadata['PROVIDERS']['name']}")
print(f"Temporal extent: {metadata['TEMPORAL_EXTENT']}")
print(f"Spatial extent: {metadata['SPATIAL_EXTENT']}")
print(f"Keywords: {metadata['KEYWORDS']}")

Collection ready: salinity_increase
Description: Description
Provider: Provider Name
Temporal extent: ['2018-01-01', '2050-01-01']
Spatial extent: [104.532771, 8.580031, 107.024243, 11.244372]
Keywords: ['Keyword 1', 'Keyword 2', 'Keyword 3']


## 4) Build one test STAC item

Validate the item-building pipeline on a single COG before running the full batch. This step exercises `process_block()` end to end — geometry, projection extension, asset hrefs, item datetime (from filename year), and STAC extensions — and surfaces configuration errors early.

The test item is not added to the collection and is not written to disk. Review the printed item id, datetime, and data asset href to confirm the output is correct before proceeding to step 5.

In [5]:
test_scenario = "cc45y"
test_tif = sorted((cog_dir / test_scenario).glob("*.tif"))[0]

test_item = process_block(
    test_tif,
    cog_dir,
    metadata,
    storage_prefix=urljoin(href_prefix, test_scenario),
)

print(f"Test item id: {test_item.id}")
print(f"Test item datetime: {test_item.datetime}")
print(f"Data asset href: {test_item.assets['data'].href}")

Test item id: cc45y/p50_2030.tif
Test item datetime: 2030-01-01 00:00:00
Data asset href: https://storage.googleapis.com/gca-data-public/gca/salinity_increase/cc45y/p50_2030.tif


## 5) Build all STAC items

Generate one STAC item for each COG across all scenario folders and register them with the collection created in step 3. Each item includes:

- Spatial metadata (geometry, bbox, datetime)
- Projection extension (native CRS, transform, shape)
- A data asset pointing to the COG on GCS
- A visual asset pointing to the WMS endpoint

This step operates entirely in memory. The collection and its items exist as Python objects until they are serialized to disk in step 6. A summary table reports the number of items created and any processing errors.

In [6]:
folder_structure = {scenario: [] for scenario in scenarios}
path_list = get_paths(folder_structure)

items = []
item_rows = []
item_errors = []

for cur_path in path_list:
    scenario_dir = cog_dir / cur_path
    if not scenario_dir.exists():
        item_errors.append({"scenario": cur_path, "error": "folder not found"})
        continue

    print(f"now working on: {cur_path}")
    for cur_tif in sorted(scenario_dir.glob("*.tif")):
        try:
            item = process_block(
                cur_tif,
                cog_dir,
                metadata,
                storage_prefix=urljoin(href_prefix, cur_path),
            )
            item_href = stac_dir / collection_id / "items" / cur_path / f"{cur_tif.stem}.json"
            item.set_self_href(str(item_href))
            items.append(item)
            collection.add_item(item)
            item_rows.append({"scenario": cur_path, "item_id": item.id, "stac_href": str(item_href)})
        except Exception as exc:
            item_errors.append({"scenario": cur_path, "file": cur_tif.name, "error": str(exc)})

print(f"Items created: {len(item_rows)}")
print(f"Errors: {len(item_errors)}")

if item_rows:
    display(pd.DataFrame(item_rows).sort_values(["scenario", "item_id"]))
if item_errors:
    display(pd.DataFrame(item_errors))

now working on: cc45sm2rb1y
now working on: cc45sm2y
now working on: cc45y
now working on: cc85sb2y
now working on: cc85y
Items created: 15
Errors: 0


,scenario,item_id,stac_href
0,cc45sm2rb1y,cc45sm2rb1y/p50_2030.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
1,cc45sm2rb1y,cc45sm2rb1y/p50_2040.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
2,cc45sm2rb1y,cc45sm2rb1y/p50_2050.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
3,cc45sm2y,cc45sm2y/p50_2030.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
4,cc45sm2y,cc45sm2y/p50_2040.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
5,cc45sm2y,cc45sm2y/p50_2050.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
6,cc45y,cc45y/p50_2030.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
7,cc45y,cc45y/p50_2040.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
8,cc45y,cc45y/p50_2050.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...
9,cc85sb2y,cc85sb2y/p50_2030.tif,C:\Ocean\Work\Projects\2025\STAC\Tools\Reposit...


## 6) Save STAC catalog locally

Serialize the in-memory collection and items to the local STAC directory. This step is distinct from item construction because it handles catalog persistence and publication layout:

1. Create item subdirectories under `STAC/data/current/{collection_id}/items/`
2. Derive the collection spatial extent from its items
3. Merge the collection into the root `catalog.json`
4. Normalize hrefs using the CoCliCo COG layout convention
5. Validate the collection and catalog

After this step, the STAC catalog is available locally and ready for inspection or cloud upload.

In [7]:
stac_io = DefaultStacIO()
layout = CoCliCoCOGLayout()

for cur_path in path_list:
    (stac_dir / collection_id / "items" / cur_path).mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
    dest_href=str(stac_dir),
    stac_io=stac_io,
)

collection.validate_all()
catalog.validate_all()
print(f"STAC saved to: {stac_dir}")

Removed existing child: salinity_increase
STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 7) Upload COGs to Google Cloud (optional)

Publish raster assets to Google Cloud Storage. Each scenario folder under `cogs/` is uploaded once to `gs://{bucket_name}/{bucket_proj}/{proj_name}/`, matching the hrefs referenced in the STAC item data assets.

Requires valid Google Cloud credentials at `google_cred_path`. Skip this step if COGs are already present on GCS.

In [8]:
load_google_credentials(google_token_fp=google_cred_path)

for cur_path in path_list:
    scenario_cog_dir = cog_dir / cur_path
    if not scenario_cog_dir.exists():
        print(f"Skipping COG upload (folder not found): {scenario_cog_dir}")
        continue

    dir_to_google_cloud(
        dir_path=str(scenario_cog_dir),
        gcs_project=gcs_project,
        bucket_name=bucket_name,
        bucket_proj=bucket_proj,
        dir_name=proj_name,
    )
    print(cur_path)

Google Application Credentials load into environment.


C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:230: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:243: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Writing to directory at gca-data-public/gca/salinity_increase...
Done!
cc45sm2rb1y
Writing to directory at gca-data-public/gca/salinity_increase...
Done!
cc45sm2y
Writing to directory at gca-data-public/gca/salinity_increase...
Done!
cc45y
Writing to directory at gca-data-public/gca/salinity_increase...
Done!
cc85sb2y
Writing to directory at gca-data-public/gca/salinity_increase...
Done!
cc85y


## 8) Upload STAC catalog to Google Cloud (optional)

Publish the local STAC catalog to Google Cloud Storage at `gs://gca-data-public/gca/{stac_cloud_name}/`. This makes the catalog JSON and item metadata publicly accessible to STAC clients and downstream applications.

Run this step only after step 6 has completed successfully and the local catalog passes validation.

In [9]:
load_google_credentials(google_token_fp=google_cred_path)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*does not advertise any conformance classes.*")
    catalog = pystac_client.Client.open(str(stac_dir / "catalog.json"))

dir_to_google_cloud(
    dir_path=str(stac_dir),
    gcs_project=gcs_project,
    bucket_name=bucket_name,
    bucket_proj=bucket_proj,
    dir_name=stac_cloud_name,
)

Google Application Credentials load into environment.
Cloud directory gca-data-public/gca/gca-stac-7 already exists...
Do you want to overwirte this directory? [y/N]:Writing to directory at gca-data-public/gca/gca-stac-7...
Done!


## 9) Publish layers to GeoServer WMS (optional)

Register uploaded COGs as external coverage stores on GeoServer so they are available through the WMS endpoint referenced in each STAC item's visual asset.

**Prerequisites:**
- Step 7 completed — COGs are present on GCS at the server mount path (`/opt/gca-data-public/gca/salinity_increase/...`)
- GeoServer credentials configured in `global-coastal-atlas/.env` (copy from `.env.example`)

**Layer naming:** `{scenario}_{probability}_{year}` (e.g. `cc45y_p50_2030` from `cogs/cc45y/p50_2030.tif`)

In [10]:
geoserver_credentials = load_geoserver_credentials(env_path)
geo = create_geoserver_client(
    geoserver_credentials["base_url"],
    geoserver_credentials["username"],
    geoserver_credentials["password"],
)

wms_published_rows, wms_error_rows = publish_layers_to_geoserver(
    geo,
    metadata,
    scenarios,
    cog_dir,
    geoserver_base_url=geoserver_credentials["base_url"],
    geoserver_username=geoserver_credentials["username"],
    geoserver_password=geoserver_credentials["password"],
    gcs_mount_prefix=geoserver_gcs_mount_prefix,
    layer_style=metadata["WMS_DATASET"],
)

print(f"WMS layers published: {len(wms_published_rows)}")
print(f"WMS errors: {len(wms_error_rows)}")

if wms_published_rows:
    display(pd.DataFrame(wms_published_rows).sort_values(["scenario", "layer_name"]))
if wms_error_rows:
    display(pd.DataFrame(wms_error_rows))

Workspace 'salinity_increase' already exists
Publishing WMS layers for scenario: cc45sm2rb1y
  Published salinity_increase:cc45sm2rb1y_p50_2030
  Published salinity_increase:cc45sm2rb1y_p50_2040
  Published salinity_increase:cc45sm2rb1y_p50_2050
Publishing WMS layers for scenario: cc45sm2y
  Published salinity_increase:cc45sm2y_p50_2030
  Published salinity_increase:cc45sm2y_p50_2040
  Published salinity_increase:cc45sm2y_p50_2050
Publishing WMS layers for scenario: cc45y
  Published salinity_increase:cc45y_p50_2030
  Published salinity_increase:cc45y_p50_2040
  Published salinity_increase:cc45y_p50_2050
Publishing WMS layers for scenario: cc85sb2y
  Published salinity_increase:cc85sb2y_p50_2030
  Published salinity_increase:cc85sb2y_p50_2040
  Published salinity_increase:cc85sb2y_p50_2050
Publishing WMS layers for scenario: cc85y
  Published salinity_increase:cc85y_p50_2030
  Published salinity_increase:cc85y_p50_2040
  Published salinity_increase:cc85y_p50_2050
WMS layers published: 

,scenario,layer_name,workspace,tif_path,status
0,cc45sm2rb1y,cc45sm2rb1y_p50_2030,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
1,cc45sm2rb1y,cc45sm2rb1y_p50_2040,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
2,cc45sm2rb1y,cc45sm2rb1y_p50_2050,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
3,cc45sm2y,cc45sm2y_p50_2030,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
4,cc45sm2y,cc45sm2y_p50_2040,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
5,cc45sm2y,cc45sm2y_p50_2050,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
6,cc45y,cc45y_p50_2030,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
7,cc45y,cc45y_p50_2040,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
8,cc45y,cc45y_p50_2050,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc4...,published
9,cc85sb2y,cc85sb2y_p50_2030,salinity_increase,/opt/gca-data-public/gca/salinity_increase/cc8...,published
